# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Apply imputation methods that match missingness patterns, feature types, and estimator requirements. 
- Use missingness indicators and empty-feature handling to preserve useful absence information. 
- Transform classification and regression targets using label, multilabel, and target-regression utilities. 


## **1. Imputation Methods**

### **1.1. Simple Imputation Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_01.jpg?v=1783834403" width="250">



>* Fill missing values with one rule.
>* Fast, clear baseline for limited missingness.

>* Choose values by feature type and distribution.
>* Missingness itself can carry useful meaning.

>* Simple imputation trades completeness for possible bias.
>* Best as baseline when speed matters.



In [ ]:
#@title Python Code - Simple Imputation Basics

# Simple imputation fills missing values consistently.
# Different feature types need different replacements.
# This example compares numeric and categorical choices.

import numpy as np
import pandas as pd

# Build a tiny dataset with missing values.
df = pd.DataFrame({
    "study_hours": [2.5, np.nan, 4.0, 3.5, np.nan],
    "income_usd": [42000, 50000, np.nan, 48000, 200000],
    "housing": ["dorm", "off-campus", np.nan, "dorm", np.nan],
})

# Show the original missing-value counts.
missing_before = df.isna().sum()

# Compute simple numeric replacement values.
study_fill = df["study_hours"].median()
income_fill = df["income_usd"].median()

# Fill numeric and categorical columns simply.
filled = df.copy()
filled["study_hours"] = filled["study_hours"].fillna(study_fill)
filled["income_usd"] = filled["income_usd"].fillna(income_fill)
filled["housing"] = filled["housing"].fillna("Unknown")

# Show the completed missing-value counts.
missing_after = filled.isna().sum()

# Compare mean and median for skewed income.
income_mean = df["income_usd"].mean()
income_median = df["income_usd"].median()

# Print a compact teaching summary.
print("Missing before:", missing_before.to_dict())
print("Missing after:", missing_after.to_dict())
print("Study hours median used:", study_fill)
print("Income mean:", round(income_mean, 1))
print("Income median used:", income_median)
print("Housing fill label:", "Unknown")
print("Filled rows 2 and 5:")
print(filled.loc[[1, 4], ["study_hours", "income_usd", "housing"]])



### **1.2. Choosing Imputation Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_02.jpg?v=1783834406" width="250">



>* Match imputation to missingness and feature type.
>* Simple fills can distort patterned missing data.

>* Estimator choice should guide imputation strategy.
>* Simple fills can distort important patterns.

>* Balance realism, cost, and interpretability.
>* Compare methods; heavy missingness reduces reliability.



### **1.3. KNN Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_03.jpg?v=1783834408" width="250">



>* Uses similar records to fill missing values.
>* Preserves local patterns better than global averages.

>* Scale and encode features for valid neighbors.
>* KNN works best with informative similarity features.

>* Costly and weaker with sparse missing data.
>* Best for meaningful local patterns and variation.



## **2. Missingness Signals**

### **2.1. Indicator Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_01.jpg?v=1783834410" width="250">



>* Indicators preserve whether values were originally missing.
>* Missingness itself can improve model understanding.

>* Missingness can reveal behavior or context.
>* Indicators preserve that signal after imputation.

>* Indicators preserve missingness beyond imputation.
>* Interpret missingness carefully for fairness and insight.



In [ ]:
#@title Python Code - Indicator Features

# Missingness indicators preserve useful absence information.
# This example adds flags before filling values.
# Small tables keep the idea easy.

import numpy as np
import pandas as pd

# Build a tiny customer table.
df = pd.DataFrame({
    "income": [52000, np.nan, 61000, np.nan, 48000],
    "loyalty": ["gold", None, "silver", None, "gold"],
})

# Create indicator features first.
df["income_was_missing"] = df["income"].isna().astype(int)
df["loyalty_was_missing"] = df["loyalty"].isna().astype(int)

# Fill missing values afterward.
df["income_filled"] = df["income"].fillna(df["income"].median())
df["loyalty_filled"] = df["loyalty"].fillna("unknown")

# Show the original missing counts.
print("Original missing counts:")
print(df[["income", "loyalty"]].isna().sum().to_string())

# Show how indicators preserve history.
print("\nIndicator columns:")
print(df[["income_was_missing", "loyalty_was_missing"]].to_string(index=False))

# Show the final filled columns.
print("\nFilled values with preserved signals:")
print(df[["income_filled", "loyalty_filled"]].to_string(index=False))



### **2.2. Empty Feature Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_02.jpg?v=1783834412" width="250">



>* Empty features need special handling decisions.
>* Dropping them may lose schema meaning.

>* Complete absence can reveal process context.
>* Keep schema and flag fully missing features.

>* Balance predictive value with pipeline stability.
>* Keep empties intentionally, with placeholders and signals.



In [ ]:
#@title Python Code - Empty Feature Handling

# This script shows empty feature handling.
# Missingness indicators can preserve useful signals.
# We use pandas only, not models.

import numpy as np
import pandas as pd

# Build a tiny example dataset.
df = pd.DataFrame({
    "age": [25, 31, 29, 40],
    "test_score": [np.nan, np.nan, np.nan, np.nan],
    "city": ["Boston", "Austin", "Boston", "Denver"],
})

# Detect columns that are completely missing.
empty_cols = df.columns[df.isna().all()].tolist()

# Keep a signal for empty features.
for col in empty_cols:
    df[f"{col}_was_empty"] = 1

# Fill empty numeric features with a placeholder.
for col in empty_cols:
    df[col] = -1

# Show the original empty columns.
print("Completely missing columns:", empty_cols)

# Show why the indicator matters.
print("Filled value in test_score:", df.loc[0, "test_score"])
print("Missingness signal:", df.loc[0, "test_score_was_empty"])

# Show the final columns kept.
print("Final columns:", df.columns.tolist())

# Preview the small transformed data.
print(df.head(2).to_string(index=False))



### **2.3. Empty Feature Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_03.jpg?v=1783834414" width="250">



>* Completely empty features need special handling.
>* Their absence can reveal structural context.

>* Empty columns can still carry context.
>* Keep or drop them deliberately.

>* Empty features can reveal meaningful context.
>* Consistent handling keeps preprocessing stable.



In [ ]:
#@title Python Code - Empty Feature Handling

# Empty columns can still carry useful meaning.
# This example preserves an all-missing feature.
# Missingness indicators help keep absence visible.

import numpy as np
import pandas as pd

# Build a tiny categorical dataset.
df = pd.DataFrame({
    "clinic": ["North", "South", "North", "East"],
    "service_note": [np.nan, np.nan, np.nan, np.nan],
    "priority": ["high", "low", "low", "high"],
})

# Check which columns are completely missing.
empty_mask = df.isna().all()
empty_columns = empty_mask[empty_mask].index.tolist()

# Preserve emptiness with an indicator column.
for col in empty_columns:
    df[f"{col}_was_empty"] = df[col].isna().astype(int)

# Fill the empty feature with a placeholder.
for col in empty_columns:
    df[col] = df[col].fillna("Not collected")

# Show the original empty columns.
print("Empty columns found:", empty_columns)

# Show why dropping could lose information.
print("Indicator values:", df["service_note_was_empty"].tolist())

# Show the preserved feature values.
print("Filled values:", df["service_note"].tolist())

# Show the final stable column set.
print("Final columns:", df.columns.tolist())



## **3. Target Transformation Tools**

### **3.1. Label Encoding Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_01.jpg?v=1783834416" width="250">



>* Converts class names into numeric target labels
>* Supports training and restores readable predictions

>* Encoded class numbers are not ordered.
>* Keep label mappings consistent across pipeline.

>* Best for single-label classification targets.
>* Not for multilabel or regression targets.



In [ ]:
#@title Python Code - Label Encoding Basics

# This script demonstrates basic target label encoding.
# It maps class names to stable integers.
# Predictions can be decoded back clearly.

# Import a simple encoder utility.
from sklearn.preprocessing import LabelEncoder

# Create a small categorical target.
y_text = ["spam", "personal", "promo", "spam", "promo"]

# Fit the encoder on target labels.
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_text)

# Check that lengths still match.
assert len(y_text) == len(y_encoded)

# Show the learned class order.
print("Original labels:", y_text)
print("Classes learned:", list(encoder.classes_))
print("Encoded target:", y_encoded.tolist())

# Decode example model outputs safely.
predicted_codes = [2, 0, 1]
predicted_labels = encoder.inverse_transform(predicted_codes)

# Display decoded predictions clearly.
print("Predicted codes:", predicted_codes)
print("Decoded labels:", predicted_labels.tolist())
print("Numbers are identifiers, not rankings.")



### **3.2. Multilabel Target Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_02.jpg?v=1783834418" width="250">



>* One item can have multiple labels.
>* Encoding uses binary positions for each label.

>* One dimension per label, multiple tags possible.
>* Consistent label vocabularies keep encoding reliable.

>* Multilabel encoding shapes interpretation and evaluation.
>* Keep label mappings clear; partial matches matter.



In [ ]:
#@title Python Code - Multilabel Target Encoding

# This script demonstrates multilabel target encoding.
# It uses small examples and readable outputs.
# The focus is target transformation basics.

import numpy as np
import pandas as pd

# Create tiny multilabel target examples.
movie_tags = [
    ("comedy", "family"),
    ("mystery",),
    ("comedy", "romance"),
    ("family", "romance", "comedy"),
]

# Build a stable label vocabulary.
all_labels = sorted({label for row in movie_tags for label in row})
label_to_index = {label: i for i, label in enumerate(all_labels)}

# Encode each observation as zeros and ones.
encoded = np.zeros((len(movie_tags), len(all_labels)), dtype=int)
for row_index, labels in enumerate(movie_tags):
    for label in labels:
        encoded[row_index, label_to_index[label]] = 1

# Validate the encoded target shape.
assert encoded.shape == (len(movie_tags), len(all_labels))
assert encoded.sum() == sum(len(row) for row in movie_tags)

# Show the encoded target neatly.
encoded_df = pd.DataFrame(encoded, columns=all_labels)
print("Label order:", all_labels)
print("Encoded shape:", encoded.shape)
print(encoded_df.to_string(index=False))

# Decode one row back to labels.
decoded_first = [all_labels[i] for i, value in enumerate(encoded[0]) if value == 1]
print("First item decoded:", decoded_first)



### **3.3. Target Regression Utilities**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_03.jpg?v=1783834420" width="250">



>* Transforms skewed numeric targets for easier learning.
>* Predictions return in original meaningful units.

>* Utilities automate target transform and reversal.
>* They improve learning, keep predictions interpretable.

>* Transform targets only when behavior warrants it.
>* Choose transformations for reliable, interpretable predictions.



# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>


In this lecture, you learned to:
- Apply imputation methods that match missingness patterns, feature types, and estimator requirements. 
- Use missingness indicators and empty-feature handling to preserve useful absence information. 
- Transform classification and regression targets using label, multilabel, and target-regression utilities. 

In the next Module (Module 6), we will go over 'None'